In [0]:
from pyspark.sql.types import StructType,StructField, StringType,IntegerType,DecimalType,DoubleType
from pyspark.sql import functions as F

In [0]:
pythonconfig = {
                "null_checks": ["email", "dept_id"], # columns that must not be null 
                "duplicate_checks": ["emp_id"], #columns that define uniqueness 
                "audit_columns": ["last_updated_ts"] #used to decide "latest" record when duplicates exist
                }

In [0]:
employee_schema = StructType([
    StructField('emp_id',IntegerType()),
    StructField('name',StringType()),
    StructField('dept_id',IntegerType()),
    StructField('email',StringType()),
    StructField('last_update_ts',StringType())
]) 

employee_data = [
    [101,"Anu",10,"anu@x.com","2024-01-01 10:00:00"],
    [101,"Anu",10,"anu.new@x.com","2024-02-01 10:00:00"],
    [102,"Ravi",None,"ravi@x.com","2024-01-05 10:00:00"],
    [103,"Neha",12,None,"2024-01-06 10:00:00"],
    [104,"Sam",13,"sam@x.com","2024-01-07 10:00:00"]
]
employee_df = spark.createDataFrame(employee_data,employee_schema)

display(employee_df)

In [0]:
null_cols = pythonconfig["null_checks"]

# A. Produce the Summary DataFrame (Calculated efficiently in a single action)
# We count rows where the column is null for every column in the config
exprs = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in null_cols]
agg_row = employee_df.select(exprs).collect()[0].asDict()

# Convert the dictionary back into a PySpark DataFrame for the summary
summary_data = [(k, v) for k, v in agg_row.items()]
summary_df = spark.createDataFrame(summary_data, ["column_name", "null_count"])

# Display the summary DataFrame
display(summary_df) # Use summary_df.show() if running locally outside Databricks

In [0]:
null_cols = pythonconfig["null_checks"]
print(null_cols)
schema= StructType([
    StructField("column_name", StringType()),
    StructField("null_count", IntegerType())
    ])

exprs = [F.count(F.when(F.col(c).isNull(), c )).alias(c) for c in null_cols]
# create a dctionar toget column names in one column
df = employee_df.select(exprs).collect()[0].asDict()
print(df)

data = [(k,v) for k,v in df.items()]

null_df = spark.createDataFrame(data, ["column_name","null_count"])

null_df.display()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count
from functools import reduce
from operator import or_

# ==========================================
# STEP 1: NULL CHECK LOGIC
# ==========================================

null_cols = config["null_checks"]

# A. Produce the Summary DataFrame (Calculated efficiently in a single action)
# We count rows where the column is null for every column in the config
exprs = [count(when(col(c).isNull(), c)).alias(c) for c in null_cols]
agg_row = employee_df.select(exprs).collect()[0].asDict()

# Convert the dictionary back into a PySpark DataFrame for the summary
summary_data = [(k, v) for k, v in agg_row.items()]
summary_df = spark.createDataFrame(summary_data, ["column_name", "null_count"])

# Display the summary DataFrame
display(summary_df) # Use summary_df.show() if running locally outside Databricks

# B. Separate the records
# Dynamically build an OR condition for all columns in null_checks
# E.g., col("email").isNull() | col("dept_id").isNull()
null_condition = reduce(or_, [col(c).isNull() for c in null_cols])

# Filter into exceptions and remaining valid records
null_exceptions_df = employee_df.filter(null_condition)
valid_df = employee_df.filter(~null_condition)

print("--- Null Exceptions DataFrame ---")
null_exceptions_df.show()

print("--- Remaining (Valid) DataFrame ---")
valid_df.show()